# Pillar-0 spatial audit: prepare transfer artifact

This CPU-only Kaggle notebook selects one report-only study, reconstructs one series per standard plane, and writes the three preprocessed FP16 volumes to this private Kaggle notebook's output. The tensors stay in Kaggle storage until the separate Modal inference step downloads them into ephemeral memory. No training or official labels are used.


In [ ]:
### Cell 1 — locate the dataset and load its tables

from pathlib import Path
import random

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Kaggle may expose competition data in either location.
data_candidates = [
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
]

DATA = next((path for path in data_candidates if path.is_dir()), None)

if DATA is None:
    raise FileNotFoundError(
        "Competition data was not found. Attach "
        "'rsna-knee-abnormality-detection' to this notebook."
    )

csv_paths = sorted(DATA.glob("*.csv"))

print("Dataset directory:", DATA)
print("CSV files:", [path.name for path in csv_paths])
print("GPU use: none; preprocessing only")

train = pd.read_csv(DATA / "train.csv")
train_series = pd.read_csv(DATA / "train_series.csv")
test = pd.read_csv(DATA / "test.csv")
test_series = pd.read_csv(DATA / "test_series.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

print()
print("train:", train.shape)
print("train_series:", train_series.shape)
print("test:", test.shape)
print("test_series:", test_series.shape)
print("sample_submission:", sample_submission.shape)



In [ ]:
### Select a report-only study with the widest plane coverage

TARGETS = [column for column in sample_submission.columns if column != "StudyInstanceUID"]
train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
train_series["StudyInstanceUID"] = train_series["StudyInstanceUID"].astype(str)
train_series["SeriesInstanceUID"] = train_series["SeriesInstanceUID"].astype(str)

report_only_ids = set(
    train.loc[train[TARGETS].isna().all(axis=1), "StudyInstanceUID"]
)
planes = {"Axial", "Coronal", "Sagittal"}
best_rows = None
best_study = None
best_score = -1

for study_uid in sorted(report_only_ids):
    group = train_series[
        (train_series["StudyInstanceUID"] == study_uid)
        & train_series["Anatomical_Plane"].isin(planes)
    ].copy()
    if group.empty:
        continue
    group = group.sort_values(
        ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"],
        ascending=[True, False, False, True],
    )
    selected = group.groupby("Anatomical_Plane", as_index=False).head(1)
    score = selected["Anatomical_Plane"].nunique()
    if score > best_score:
        best_rows, best_study, best_score = selected, study_uid, score
    if score == 3:
        break

if best_rows is None or best_score < 2:
    raise RuntimeError("Could not find a report-only study with at least two standard planes")

example_study_uid = best_study
example_series = best_rows.sort_values("Anatomical_Plane").reset_index(drop=True)
representative_rows = example_series
print("Report-only study selected; no official labels are read.")
print("Series selected by plane:")
display(example_series[["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]])


In [ ]:
import pydicom
import scipy
print('Using preinstalled pydicom', pydicom.__version__, 'and scipy', scipy.__version__)

import pydicom
import matplotlib.pyplot as plt
import torch

TRAIN_DICOM_ROOT = DATA / "train_series"
assert TRAIN_DICOM_ROOT.is_dir(), TRAIN_DICOM_ROOT


In [ ]:
### Cell 5 — reconstruct and visualize one real volume

import matplotlib.pyplot as plt


def reconstruct_series(study_uid, series_uid):
  """
  Reconstruct one classic single-frame MRI series.

  Returns
  -------
  volume:
      Float32 NumPy array with axes (slice, row, column).
  geometry:
      Spatial metadata describing the reconstructed array.
  """
  folder = (
      TRAIN_DICOM_ROOT
      / str(study_uid)
      / str(series_uid)
  )

  dicom_paths = sorted(folder.glob("*.dcm"))

  if len(dicom_paths) < 2:
      raise ValueError(
          f"Expected at least two DICOM slices: {folder}"
      )

  headers = [
      pydicom.dcmread(path, stop_before_pixels=True)
      for path in dicom_paths
  ]

  reference_orientation = np.asarray(
      headers[0].ImageOrientationPatient,
      dtype=np.float64,
  )

  orientations = np.asarray(
      [
          header.ImageOrientationPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  if not np.allclose(
      orientations,
      reference_orientation,
      atol=1e-4,
      rtol=0,
  ):
      raise ValueError("Orientation varies within the series")

  pixel_spacing = np.asarray(
      headers[0].PixelSpacing,
      dtype=np.float64,
  )

  for header in headers:
      current_spacing = np.asarray(
          header.PixelSpacing,
          dtype=np.float64,
      )

      if not np.allclose(
          current_spacing,
          pixel_spacing,
          atol=1e-4,
          rtol=0,
      ):
          raise ValueError(
              "PixelSpacing varies within the series"
          )

  # DICOM IOP stores the directions across columns and rows.
  column_axis_direction = reference_orientation[:3]
  row_axis_direction = reference_orientation[3:]

  slice_normal = np.cross(
      column_axis_direction,
      row_axis_direction,
  )

  positions = np.asarray(
      [
          header.ImagePositionPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  projections = positions @ slice_normal
  physical_order = np.argsort(
      projections,
      kind="stable",
  )

  ordered_positions = positions[physical_order]
  ordered_projections = projections[physical_order]
  slice_gaps = np.diff(ordered_projections)

  if np.any(slice_gaps <= 1e-3):
      raise ValueError(
          "Repeated or indistinguishable slice positions"
      )

  slice_spacing = float(np.median(slice_gaps))

  if not np.allclose(
      slice_gaps,
      slice_spacing,
      atol=0.05,
      rtol=0.01,
  ):
      raise ValueError(
          "Irregular spacing or missing slices"
      )

  ordered_slices = []
  ordered_files = []

  for original_index in physical_order:
      path = dicom_paths[original_index]
      dcm = pydicom.dcmread(path)

      pixels = dcm.pixel_array.astype(np.float32)

      if pixels.shape != (
          int(dcm.Rows),
          int(dcm.Columns),
      ):
          raise ValueError(
              f"Unexpected pixel shape: {pixels.shape}"
          )

      # Exclude declared padding from intensity statistics.
      if hasattr(dcm, "PixelPaddingValue"):
          padding_value = float(dcm.PixelPaddingValue)

          if hasattr(dcm, "PixelPaddingRangeLimit"):
              padding_limit = float(
                  dcm.PixelPaddingRangeLimit
              )
          else:
              padding_limit = padding_value

          padding_low = min(
              padding_value,
              padding_limit,
          )
          padding_high = max(
              padding_value,
              padding_limit,
          )

          padding_mask = (
              (pixels >= padding_low)
              & (pixels <= padding_high)
          )
      else:
          padding_mask = np.zeros(
              pixels.shape,
              dtype=bool,
          )

      slope = float(
          getattr(dcm, "RescaleSlope", 1.0)
      )
      intercept = float(
          getattr(dcm, "RescaleIntercept", 0.0)
      )

      pixels = pixels * slope + intercept
      pixels[padding_mask] = np.nan

      if not np.isfinite(pixels).any():
          raise ValueError(
              f"No finite pixels in {path.name}"
          )

      ordered_slices.append(pixels)
      ordered_files.append(str(path))

  volume = np.stack(
      ordered_slices,
      axis=0,
  ).astype(np.float32)

  # Array axes are (slice, row, column).
  # This affine maps those indices into DICOM LPS millimetres.
  affine = np.eye(4, dtype=np.float64)
  affine[:3, 0] = (
      slice_normal * slice_spacing
  )
  affine[:3, 1] = (
      row_axis_direction * pixel_spacing[0]
  )
  affine[:3, 2] = (
      column_axis_direction * pixel_spacing[1]
  )
  affine[:3, 3] = ordered_positions[0]

  geometry = {
      "array_axes": ("slice", "row", "column"),
      "coordinate_system": "DICOM LPS millimetres",
      "spacing_drc_mm": (
          slice_spacing,
          float(pixel_spacing[0]),
          float(pixel_spacing[1]),
      ),
      "affine_array_to_lps": affine,
      "slice_normal_lps": slice_normal,
      "ordered_positions_lps": ordered_positions,
      "ordered_files": ordered_files,
      "photometric_interpretation": str(
          getattr(
              headers[0],
              "PhotometricInterpretation",
              "",
          )
      ),
  }

  return volume, geometry




In [ ]:
### Cell 6 — compare sagittal, coronal, and axial reconstructions

# Prefer a fluid-sensitive series for each anatomical plane.
representative_rows = (
  example_series
  .sort_values(
      ["Anatomical_Plane", "Fluid_Sensitive"],
      ascending=[True, False],
  )
  .groupby("Anatomical_Plane", as_index=False)
  .head(1)
  .sort_values("Anatomical_Plane")
  .reset_index(drop=True)
)

fig, axes = plt.subplots(
  len(representative_rows),
  3,
  figsize=(13, 4 * len(representative_rows)),
)

reconstruction_summary = []

for row_index, series_row in representative_rows.iterrows():
  plane = series_row["Anatomical_Plane"]

  plane_volume, plane_geometry = reconstruct_series(
      series_row["StudyInstanceUID"],
      series_row["SeriesInstanceUID"],
  )

  (
      plane_slice_spacing,
      plane_row_spacing,
      plane_column_spacing,
  ) = plane_geometry["spacing_drc_mm"]

  finite = plane_volume[
      np.isfinite(plane_volume)
  ]

  window_low, window_high = np.percentile(
      finite,
      [1, 99],
  )

  middle_slice = plane_volume.shape[0] // 2
  middle_row = plane_volume.shape[1] // 2
  middle_column = plane_volume.shape[2] // 2

  # Native acquisition plane.
  axes[row_index, 0].imshow(
      plane_volume[middle_slice],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_row_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 0].set_title(
      f"{plane}: native centre slice"
  )
  axes[row_index, 0].set_xlabel("column")
  axes[row_index, 0].set_ylabel("row")

  # Orthogonal centre-row view.
  axes[row_index, 1].imshow(
      plane_volume[:, middle_row, :],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 1].set_title(
      f"{plane}: centre row"
  )
  axes[row_index, 1].set_xlabel("column")
  axes[row_index, 1].set_ylabel(
      "physically ordered slice"
  )

  # Orthogonal centre-column view.
  axes[row_index, 2].imshow(
      plane_volume[:, :, middle_column],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_row_spacing
      ),
  )
  axes[row_index, 2].set_title(
      f"{plane}: centre column"
  )
  axes[row_index, 2].set_xlabel("row")
  axes[row_index, 2].set_ylabel(
      "physically ordered slice"
  )

  reconstruction_summary.append(
      {
          "plane": plane,
          "fluid_sensitive":
              series_row["Fluid_Sensitive"],
          "fat_suppression":
              series_row["Fat_Suppression"],
          "shape_drc": tuple(
              plane_volume.shape
          ),
          "spacing_drc_mm":
              plane_geometry["spacing_drc_mm"],
          "physical_fov_mm": tuple(
              round(size * spacing, 2)
              for size, spacing in zip(
                  plane_volume.shape,
                  plane_geometry[
                      "spacing_drc_mm"
                  ],
              )
          ),
      }
  )

plt.tight_layout()
plt.show()

display(
  pd.DataFrame(reconstruction_summary)
)

## Visual review gate

Inspect the reconstruction figure in the preceding cell before continuing. If any view is blank, truncated, or incorrectly oriented, stop and fix preprocessing first. Run the remaining cells individually after confirming the images are usable. The selected study is report-only; the notebook never reads official labels.


In [ ]:
### Cell 10 — canonicalize and preprocess one series

from itertools import product

from scipy.ndimage import affine_transform


TARGET_SPACING_XYZ_MM = np.array(
  [1.0, 1.0, 1.0],
  dtype=np.float64,
)

TARGET_SHAPE_ZYX = (
  192,
  384,
  384,
)


def canonical_resample_lps(
  native_volume,
  native_affine,
  target_spacing_xyz_mm=TARGET_SPACING_XYZ_MM,
):
  """Resample signal and validity together, avoiding padding bleed.

  Returns intensity normalized by interpolated support, a fractional support
  map in [0,1], and canonical-grid metadata. A genuine zero remains supported.
  """
  native_shape = np.asarray(native_volume.shape, dtype=int)
  native_validity = np.isfinite(native_volume).astype(np.float32)
  native_signal = np.where(
      native_validity > 0,
      np.nan_to_num(native_volume, nan=0.0, posinf=0.0, neginf=0.0),
      0.0,
  ).astype(np.float32)

  native_corners = np.asarray(list(product(
      [0, native_shape[0] - 1],
      [0, native_shape[1] - 1],
      [0, native_shape[2] - 1],
  )), dtype=np.float64)
  homogeneous_corners = np.concatenate(
      [native_corners, np.ones((len(native_corners), 1), dtype=np.float64)], axis=1
  )
  lps_corners = (native_affine @ homogeneous_corners.T).T[:, :3]
  lps_min = lps_corners.min(axis=0)
  lps_max = lps_corners.max(axis=0)
  physical_extent_xyz = lps_max - lps_min
  output_size_xyz = np.ceil(physical_extent_xyz / target_spacing_xyz_mm).astype(int) + 1
  output_shape_zyx = tuple(output_size_xyz[::-1])

  canonical_affine = np.eye(4, dtype=np.float64)
  canonical_affine[:3, 0] = [0, 0, target_spacing_xyz_mm[2]]
  canonical_affine[:3, 1] = [0, target_spacing_xyz_mm[1], 0]
  canonical_affine[:3, 2] = [target_spacing_xyz_mm[0], 0, 0]
  canonical_affine[:3, 3] = lps_min
  output_to_input = np.linalg.inv(native_affine) @ canonical_affine
  transform = dict(
      matrix=output_to_input[:3, :3],
      offset=output_to_input[:3, 3],
      output_shape=output_shape_zyx,
      order=1,
      mode="constant",
      cval=0.0,
      prefilter=False,
  )

  numerator = affine_transform(native_signal, **transform).astype(np.float32)
  support = affine_transform(native_validity, **transform).astype(np.float32)
  support = np.clip(support, 0.0, 1.0)
  canonical_volume = np.full(output_shape_zyx, np.nan, dtype=np.float32)
  supported = support > 1e-6
  canonical_volume[supported] = numerator[supported] / support[supported]

  metadata = {
      "input_shape_src": tuple(native_volume.shape),
      "output_shape_zyx": tuple(canonical_volume.shape),
      "output_spacing_xyz_mm": tuple(target_spacing_xyz_mm.tolist()),
      "output_origin_lps": tuple(lps_min.tolist()),
      "physical_extent_xyz_mm": tuple(physical_extent_xyz.tolist()),
      "canonical_affine_zyx_to_lps": canonical_affine,
  }
  return canonical_volume, support, metadata


def high_contrast_mr_window(
  volume,
  support,
  lower_percentile=20,
  upper_percentile=99,
  support_threshold=0.5,
):
  """Window supported voxels only; set unsupported output voxels to zero."""
  tensor = torch.as_tensor(volume, dtype=torch.float32)
  support_tensor = torch.as_tensor(support, dtype=torch.float32)
  if tensor.shape != support_tensor.shape:
      raise ValueError(f"Volume/support shape mismatch: {tensor.shape} vs {support_tensor.shape}")
  core = torch.isfinite(tensor) & (support_tensor >= support_threshold)
  finite_values = tensor[core]
  if finite_values.numel() == 0:
      raise ValueError("No supported voxels available for MRI intensity window")
  lower = torch.quantile(finite_values, lower_percentile / 100)
  upper = torch.quantile(finite_values, upper_percentile / 100)
  if upper <= lower:
      raise ValueError("Degenerate MRI intensity window")
  safe_tensor = torch.where(torch.isfinite(tensor), tensor, lower)
  windowed = ((safe_tensor - lower) / (upper - lower)).clamp(0, 1)
  has_support = torch.isfinite(tensor) & (support_tensor > 1e-6)
  windowed = torch.where(has_support, windowed, torch.zeros_like(windowed))
  return windowed, float(lower), float(upper)


def center_pad_or_crop_zyx(
  volume,
  target_shape_zyx,
):
  """
  Centre pad or crop a (z,y,x) tensor.

  Returns the processed volume and a record of each transformation.
  """
  output = torch.zeros(
      target_shape_zyx,
      dtype=volume.dtype,
  )

  source_slices = []
  destination_slices = []
  transformations = []

  for axis_name, current, target in zip(
      ("z", "y", "x"),
      volume.shape,
      target_shape_zyx,
  ):
      if current <= target:
          source_start = 0
          source_end = current

          destination_start = (
              target - current
          ) // 2
          destination_end = (
              destination_start + current
          )

          transformations.append(
              {
                  "axis": axis_name,
                  "operation": "pad",
                  "before": destination_start,
                  "after": (
                      target
                      - destination_end
                  ),
              }
          )
      else:
          source_start = (
              current - target
          ) // 2
          source_end = (
              source_start + target
          )

          destination_start = 0
          destination_end = target

          transformations.append(
              {
                  "axis": axis_name,
                  "operation": "crop",
                  "start": source_start,
                  "end": source_end,
              }
          )

      source_slices.append(
          slice(
              source_start,
              source_end,
          )
      )
      destination_slices.append(
          slice(
              destination_start,
              destination_end,
          )
      )

  output[
      tuple(destination_slices)
  ] = volume[
      tuple(source_slices)
  ]

  return output, transformations




## Pack the three preprocessed volumes

The archive contains only FP16 model inputs, low-resolution source-support fractions, and plane names. It contains no study/series identifiers or labels. The Modal job retrieves this small-kernel artifact from Kaggle, performs frozen FP32 inference, and deletes its ephemeral copy after loading.


In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F

OUTPUT_DIR = Path("/kaggle/working/pillar0_modal_transfer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

expected_planes = {"Axial", "Coronal", "Sagittal"}
actual_planes = set(representative_rows["Anatomical_Plane"].astype(str))
if actual_planes != expected_planes or len(representative_rows) != 3:
    raise RuntimeError(f"Expected exactly three standard planes; found {sorted(actual_planes)}")

volumes = []
supports = []
planes = []
support_fractions = []
native_finite_fractions = []

for row in representative_rows.itertuples(index=False):
    plane = str(row.Anatomical_Plane)
    print(f"Preparing {plane} volume on Kaggle CPU", flush=True)
    native_volume, native_geometry = reconstruct_series(
        row.StudyInstanceUID, row.SeriesInstanceUID
    )
    native_finite_fractions.append(float(np.isfinite(native_volume).mean()))
    canonical, support_canonical, _ = canonical_resample_lps(
        native_volume, native_geometry["affine_array_to_lps"]
    )
    windowed, window_low, window_high = high_contrast_mr_window(
        canonical, support_canonical
    )
    padded, _ = center_pad_or_crop_zyx(windowed, TARGET_SHAPE_ZYX)
    support_padded, _ = center_pad_or_crop_zyx(
        torch.from_numpy(support_canonical), TARGET_SHAPE_ZYX
    )

    # Existing verified checkpoint input convention: [C,H,W,D] = [3,384,384,192].
    model_hwd = padded.permute(1, 2, 0).contiguous()
    support_hwd = support_padded.permute(1, 2, 0).contiguous().clamp_(0, 1)
    support_grid = F.adaptive_avg_pool3d(
        support_hwd[None, None].float(), (64, 32, 16)
    )[0, 0]

    volume_np = model_hwd.numpy().astype(np.float16)
    support_np = support_grid.numpy().astype(np.float16)
    if volume_np.shape != (384, 384, 192) or support_np.shape != (64, 32, 16):
        raise RuntimeError(f"Unexpected {plane} tensor shapes: {volume_np.shape}, {support_np.shape}")
    if not np.isfinite(volume_np).all() or not np.isfinite(support_np).all():
        raise RuntimeError(f"Non-finite value in {plane} tensor")

    volumes.append(volume_np)
    supports.append(support_np)
    planes.append(plane)
    support_fractions.append(float(support_np.mean()))
    del native_volume, canonical, support_canonical, windowed, padded, support_padded

archive_path = OUTPUT_DIR / "pillar0_transfer.npz"
np.savez_compressed(
    archive_path,
    volumes=np.stack(volumes),
    supports=np.stack(supports),
    planes=np.asarray(planes),
)
manifest = {
    "scope": "three report-only series; preprocessed inputs only; no labels or identifiers",
    "preprocessing": "mask-aware linear resampling; intensity window estimated from voxels with support >= 0.5; unsupported voxels zeroed after normalization",
    "support_is_encoder_input": False,
    "planes": planes,
    "volume_shape_per_plane": [384, 384, 192],
    "volume_dtype": "float16",
    "support_shape_per_plane": [64, 32, 16],
    "support_dtype": "float16",
    "valid_source_support_fraction": support_fractions,
    "native_finite_voxel_fraction": native_finite_fractions,
    "native_nan_or_infinite_voxel_fraction": [1.0 - value for value in native_finite_fractions],
    "archive_bytes": archive_path.stat().st_size,
}
(OUTPUT_DIR / "transfer_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2), flush=True)
print("Kaggle output artifact ready:", archive_path, flush=True)
